# Pocket-TTS Multilingual & Multi-Speaker Fine-Tuning
This notebook fine-tunes **Pocket-TTS** (Kyutai's CALM architecture: 24-layer Flow Matching Transformer + Mimi Neural Codec) on custom African language speech datasets with speaker conditioning:
- `crestai/waxal_tts` (Waxal)
- `crestai/kin_tts` (Kinyarwanda)
- `crestai/salt_tts` (Salt / Luganda)
- `crestai/pidjin_tts` (Nigerian Pidgin)
- `crestai/wolof_tts` (Wolof)
- `crestai/twi_akosua_female_speaker_tts` (Twi)
- `crestai/vo_tts` (Vo / Ewe)

In [ ]:
# 1. Install & Verify Dependencies
!pip install torch torchaudio soundfile datasets sentencepiece huggingface_hub safetensors typer pydantic tqdm scipy
!pip install einops einx

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Setup Paths & Configuration
import os
import sys
from pathlib import Path

POCKET_TTS_DIR = Path(".").resolve() if Path("pocket_tts").exists() else Path("pocket-tts").resolve()
if str(POCKET_TTS_DIR) not in sys.path:
    sys.path.insert(0, str(POCKET_TTS_DIR))

DATA_DIR = Path("data/custom_dataset")
AUDIO_DIR = DATA_DIR / "audio"
SPEAKER_SAMPLES_DIR = DATA_DIR / "speaker_samples"
CONFIG_PATH = POCKET_TTS_DIR / "training/configs/finetune_custom_languages.yaml"

HF_TOKEN = ""  # Fill with your Hugging Face write token
HF_REPO_NAME = "crestai/pocket_tts_multilingual_finetuned"

DATASETS_CONFIG = [
    {"repo": "crestai/waxal_tts", "lang": "waxal"},
    {"repo": "crestai/kin_tts", "lang": "kinyarwanda"},
    {"repo": "crestai/salt_tts", "lang": "salt"},
    {"repo": "crestai/pidjin_tts", "lang": "pidgin"},
    {"repo": "crestai/wolof_tts", "lang": "wolof"},
    {"repo": "crestai/twi_akosua_female_speaker_tts", "lang": "twi"},
    {"repo": "crestai/vo_tts", "lang": "vo"},
]

In [ ]:
# 3. Download, Prepare Audio Manifests & Extract Speaker Catalog
import json
import random
import shutil
import soundfile as sf
from datasets import load_dataset, Audio

DATA_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
SPEAKER_SAMPLES_DIR.mkdir(parents=True, exist_ok=True)

all_records = []
TARGET_SR = 24000

for item in DATASETS_CONFIG:
    repo = item["repo"]
    lang = item["lang"]
    print(f"Processing dataset {repo} ({lang})...")
    try:
        ds = load_dataset(repo)
        split = ds["train"] if "train" in ds else list(ds.values())[0]
        split = split.cast_column("audio", Audio(sampling_rate=TARGET_SR))
        
        for idx, row in enumerate(split):
            audio_info = row.get("audio")
            if not audio_info:
                continue
            arr = audio_info["array"]
            sr = audio_info["sampling_rate"]
            text = row.get("text", "").strip()
            speaker = row.get("speaker_id", f"{lang}_speaker")
            if not text:
                continue
                
            dur = len(arr) / sr
            if dur < 1.0 or dur > 30.0:
                continue
                
            fname = f"{lang}_{str(speaker).replace(' ', '_')}_{idx:06d}.wav"
            wav_path = AUDIO_DIR / fname
            if not wav_path.exists():
                sf.write(str(wav_path), arr, sr, subtype="PCM_16")
                
            all_records.append({
                "path": str(wav_path.resolve()),
                "start": 0.0,
                "duration": float(dur),
                "transcript": text,
                "speaker": str(speaker),
                "language": lang
            })
    except Exception as e:
        print(f"Error processing {repo}: {e}")

print(f"Total utterances prepared: {len(all_records)}")

# Create Speaker Reference Catalog
speaker_catalog = {}
for r in all_records:
    key = f"{r['language']}_{r['speaker']}"
    if key not in speaker_catalog and 3.0 <= r["duration"] <= 12.0:
        ref_wav = SPEAKER_SAMPLES_DIR / f"{key}_ref.wav"
        shutil.copyfile(r["path"], ref_wav)
        speaker_catalog[key] = {
            "language": r["language"],
            "speaker_id": r["speaker"],
            "reference_audio_path": str(ref_wav.resolve()),
            "sample_transcript": r["transcript"]
        }

catalog_path = DATA_DIR / "speakers_catalog.json"
with open(catalog_path, "w", encoding="utf-8") as f:
    json.dump(speaker_catalog, f, indent=2, ensure_ascii=False)

print(f"Speaker Catalog created with {len(speaker_catalog)} distinct speakers: {catalog_path}")

random.seed(42)
random.shuffle(all_records)
n_val = max(1, int(len(all_records) * 0.02))
train_records = all_records[n_val:]
val_records = all_records[:n_val]

train_jsonl = DATA_DIR / "train_aligned.jsonl"
valid_jsonl = DATA_DIR / "valid_aligned.jsonl"

with open(train_jsonl, "w", encoding="utf-8") as f:
    for r in train_records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

with open(valid_jsonl, "w", encoding="utf-8") as f:
    for r in val_records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Manifests written: {train_jsonl} ({len(train_records)} rows), {valid_jsonl} ({len(val_records)} rows)")

In [ ]:
# 4. Train Multilingual SentencePiece Tokenizer
import sentencepiece as spm

tok_prefix = str(DATA_DIR / "tokenizer")
corpus_txt = DATA_DIR / "corpus.txt"

with open(corpus_txt, "w", encoding="utf-8") as f_out:
    for r in all_records:
        f_out.write(r["transcript"] + "\n")

spm.SentencePieceTrainer.train(
    input=str(corpus_txt),
    model_prefix=tok_prefix,
    vocab_size=4000,
    character_coverage=1.0,
    model_type="bpe",
    pad_id=-1,
    unk_id=0,
    bos_id=1,
    eos_id=2,
)

corpus_txt.unlink(missing_ok=True)
sp = spm.SentencePieceProcessor(model_file=tok_prefix + ".model")
print(f"Tokenizer trained successfully. Vocab size: {sp.get_piece_size()}")

In [ ]:
# 5. Precompute Mimi Codec Latents (High Training Throughput)
from training.scripts.precompute_latents import run_precompute

device = "cuda" if torch.cuda.is_available() else "cpu"
run_precompute(
    manifest=train_jsonl,
    device=device,
    batch_size=16,
)
print("Mimi latents precomputation complete!")

In [ ]:
# 6. Launch Training
from pocket_tts_finetune import run_training

run_training(str(CONFIG_PATH))

In [ ]:
# 7. Test Generation & Voice Cloning per Custom Speaker from Catalog
import json
from pocket_tts.models.tts_model import TTSModel
import IPython.display as ipd

checkpoint_path = "runs/finetune_custom_languages/model.safetensors"
catalog_file = DATA_DIR / "speakers_catalog.json"

with open(catalog_file, "r", encoding="utf-8") as f:
    speakers = json.load(f)

print(f"Found {len(speakers)} custom speakers in catalog:")
for key, spk in list(speakers.items())[:10]:
    print(f"- Speaker: {spk['speaker_id']} (Language: {spk['language']})")
    print(f"  Ref Audio: {spk['reference_audio_path']}")
    print(f"  Sample Text: {spk['sample_transcript']}\n")

# To generate speech with a custom speaker:
# model = TTSModel.load_model(config=str(CONFIG_PATH), checkpoint=checkpoint_path)
# target_speaker = list(speakers.values())[0]
# state = model.get_state_for_audio_prompt(target_speaker["reference_audio_path"])
# audio = model.generate_audio(state, target_speaker["sample_transcript"])
# ipd.display(ipd.Audio(audio.numpy(), rate=24000))

In [ ]:
# 8. Push Checkpoints to Hugging Face Hub
from pocket_tts_finetune import push_to_huggingface

if HF_TOKEN:
    push_to_huggingface(
        run_dir=Path("runs/finetune_custom_languages"),
        repo_name=HF_REPO_NAME,
        hf_token=HF_TOKEN,
    )
else:
    print("Please provide HF_TOKEN in Cell 2 to push to Hugging Face Hub.")